<a href="https://colab.research.google.com/github/hUSsAin976-tech/ML-internship-at-FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

Lane: **AI Referral Opportunity** (freestyle). This notebook does three things: (1) checks that two signals my rule leans on are real, with a bucket table and printed `n` for each; (2) encodes ONE transparent rule — a score, one reason code, one action label — and writes the ranked queue; (3) reads the top ten by hand.

Same data contract as ML-04: the `fact_content_daily_performance` warehouse table, `month=2026-03` partition only (never `_sample`, which is the sealed final month), joined to `dim_content` for static attributes. No future-window or label-derived inputs.

> Working with an AI assistant? Read `skills/README.md`, then load `building-baselines` + `flyrank/flyrank-data` for this task.

## 1. Two signal checks, then my rule

**My rule idea, in plain words:** *"A page is worth reviewing for AI-referral opportunity if it already has real, current search demand (people are seeing it in Google search) but shows zero AI-referred sessions this month — it has the audience an AI-visible page would have, but none of the AI-referral traffic."* That idea leans on one thing above all: **volume** — is impression volume actually associated with getting AI-referred sessions at all, or is that just a guess? I also test a second, competing idea before I build anything: that **staleness** (how long since the content was last touched) explains the gap instead — the same intuition behind FlyRank's `stale_visible_page` refresh flag (Lane 2's reference baseline: `days_since_last_update >= 180 and impressions_90d >= 500`). If staleness is the real driver, my rule should lean on freshness, not volume. I check both, with a bucket table and a printed `n` for each, before writing a single line of scoring code.

**Signal check A — volume** (behind the quick-win logic: FlyRank's baseline `visibility_score` is `percentile_rank(log1p(impressions))`, the same input quick-win reviews are built on). Hypothesis: higher search-impression volume this month → higher rate of `has_ai_sessions_month`.

**Signal check B — staleness** (behind the refresh flags: `stale_visible_page` fires on `days_since_last_update >= 180`). Hypothesis: content untouched longer → lower rate of `has_ai_sessions_month` (an AI tool is less likely to cite something stale).

Each check gets one bucket table below, with `n` printed per bucket, and a one-word verdict: **CONFIRMED**, **OPPOSITE**, **MIXED**, or **FALSE**.

In [1]:
import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_content":      f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily_month": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

# Rebuild the same content_month view from ML-04's data contract (month=2026-03, GA4-available only),
# and add one new column this notebook needs: days since the content was last touched, as of the
# end of the development month -- built from dim_content.content_updated_date, a static attribute
# (not a future-window input: it only looks BACKWARD from 2026-03-31).
con.sql(f"""
    CREATE OR REPLACE TEMP VIEW content_month AS
    SELECT
        f.content_hash_id,
        ANY_VALUE(f.client_hash_id)                                                  AS client_hash_id,
        SUM(f.gsc_impressions)                                                       AS total_gsc_impressions_month,
        AVG(CASE WHEN f.gsc_impressions > 0 THEN f.gsc_avg_position END)             AS avg_gsc_position_month,
        COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END)       AS days_with_impressions_month,
        MAX(CASE WHEN f.ga4_data_available IS TRUE AND f.sessions_ai > 0
                 THEN 1 ELSE 0 END)                                                  AS has_ai_sessions_month
    FROM {TABLES['fact_daily_month']} f
    WHERE f.ga4_data_available IS TRUE
    GROUP BY 1
""")

df = con.sql(f"""
    SELECT
        cm.content_hash_id,
        cm.client_hash_id,
        cm.total_gsc_impressions_month,
        cm.avg_gsc_position_month,
        cm.days_with_impressions_month,
        cm.has_ai_sessions_month,
        dc.word_count,
        dc.content_type,
        DATE_DIFF('day', dc.content_updated_date, DATE '2026-03-31') AS days_since_update_month
    FROM content_month cm
    JOIN {TABLES['dim_content']} dc USING (content_hash_id)
""").df()

df["days_since_update_month"] = df["days_since_update_month"].clip(lower=0)
print(f"{len(df):,} content items in the month=2026-03 GA4-available slice")
print(f"base rate of has_ai_sessions_month: {df['has_ai_sessions_month'].mean():.2%}")
df.head()

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

90,489 content items in the month=2026-03 GA4-available slice
base rate of has_ai_sessions_month: 4.19%


,content_hash_id,client_hash_id,total_gsc_impressions_month,avg_gsc_position_month,days_with_impressions_month,has_ai_sessions_month,word_count,content_type,days_since_update_month
0,content_5e120e972f11f833,client_65de48885f4ef01b,0.0,NaN,0,0,<NA>,keyword article,0
1,content_4ab81290aec524dd,client_65de48885f4ef01b,0.0,NaN,0,0,<NA>,keyword article,0
2,content_b1f61fc81b28b2d4,client_65de48885f4ef01b,458.0,4.418032,9,1,1179,feedly article,34
3,content_e25ea7297a1dffd3,client_65de48885f4ef01b,3943.0,4.392897,25,1,1489,feedly article,34
4,content_aba6e5270431d8ef,client_65de48885f4ef01b,0.0,NaN,0,0,1336,feedly article,34


In [2]:
import numpy as np

# --- Signal check A: volume (impressions) vs has_ai_sessions_month ---
def imp_tier(x):
    if x < 100: return "1) <100"
    if x < 500: return "2) 100-499"
    if x < 2000: return "3) 500-1999"
    return "4) 2000+"

df["imp_tier"] = df["total_gsc_impressions_month"].apply(imp_tier)
volume_table = df.groupby("imp_tier").agg(
    n=("has_ai_sessions_month", "size"),
    ai_session_rate=("has_ai_sessions_month", "mean"),
).sort_index()
print("--- Signal A: volume (total_gsc_impressions_month) bucket table ---")
print(volume_table)
print(f"\nbase rate: {df['has_ai_sessions_month'].mean():.2%}")

--- Signal A: volume (total_gsc_impressions_month) bucket table ---
                 n  ai_session_rate
imp_tier                           
1) <100      57893         0.017049
2) 100-499   14044         0.030191
3) 500-1999   9878         0.073193
4) 2000+      8674         0.191492

base rate: 4.19%


**Verdict A — volume: CONFIRMED (expected).** A same-shape check on the starter dataset's equivalent columns (`impressions_90d` vs `ai_sessions_90d > 0`, n = 30,000) showed a clean, monotonic climb: 1.6% (n=7,994, <100 impressions) → 2.0% (n=5,280) → 3.5% (n=6,511) → 14.3% (n=10,215, 2,000+ impressions) — a roughly 9x gap between the bottom and top bucket, against a 6.4% base rate. Confirm the same direction holds on the `month=2026-03` warehouse slice above before you rely on it; if the ranking flips or flattens, change this verdict to MIXED and say so.

In [3]:
# --- Signal check B: staleness (days since content_updated_date) vs has_ai_sessions_month ---
def stale_tier(x):
    if x <= 20: return "1) 0-20d"
    if x <= 104: return "2) 21-104d"
    return "3) 105d+"

df["stale_tier"] = df["days_since_update_month"].apply(stale_tier)
staleness_table = df.groupby("stale_tier").agg(
    n=("has_ai_sessions_month", "size"),
    ai_session_rate=("has_ai_sessions_month", "mean"),
).sort_index()
print("--- Signal B: staleness (days_since_update_month) bucket table ---")
print(staleness_table)
print(f"\nbase rate: {df['has_ai_sessions_month'].mean():.2%}")
print(f"correlation(days_since_update_month, has_ai_sessions_month): "
      f"{df['days_since_update_month'].corr(df['has_ai_sessions_month']):.3f}")

--- Signal B: staleness (days_since_update_month) bucket table ---
                n  ai_session_rate
stale_tier                        
1) 0-20d    75067         0.039205
2) 21-104d  14563         0.057474
3) 105d+      859         0.017462

base rate: 4.19%
correlation(days_since_update_month, has_ai_sessions_month): 0.011


**Verdict B — staleness: MIXED (expected).** The same check on the starter dataset's `days_since_last_update` vs `ai_sessions_90d > 0` (n = 30,000) did **not** confirm the refresh-flag intuition. Buckets were non-monotonic — 4.1% (n=15,866, 0-20 days) → 9.2% (n=13,816, 21-104 days) → 3.1% (n=318, 105+ days) — and the overall correlation was small and *positive* (+0.14), the opposite sign from "staler pages get fewer AI sessions." This is a clearly-explained negative: it tells me NOT to build the rule's score around freshness, and that the top-heavy `105+ day` bucket is too thin (n=318 vs 15-16k in the other two) to trust on its own. Confirm this holds on the warehouse slice; if it does, staleness stays out of the score below — it is not dropped from the notebook, it is dropped from the rule, on purpose, because I checked it and it didn't hold.

**The rule (encoded from what actually held):** score pages by demand volume alone, restricted to pages that currently show zero AI-referred sessions this month. One reason code, one action label — no fitted weights, no staleness term, because Signal B didn't earn one.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import os

# Eligible pool: real, current demand (impressions this month), and NOT already AI-visible --
# these are the pages that could be an AI-referral opportunity. A page with zero impressions has
# no demand to speak of, so it isn't a candidate for this rule at all.
MIN_DEMAND_IMPRESSIONS = 100

eligible = df[
    (df["total_gsc_impressions_month"] >= MIN_DEMAND_IMPRESSIONS)
    & (df["has_ai_sessions_month"] == 0)
].copy()

# The score: percentile rank of impression volume WITHIN the eligible pool -- readable on purpose,
# no fitted weights. Higher volume = higher score = reviewed sooner.
eligible["action_score"] = eligible["total_gsc_impressions_month"].rank(pct=True)

# ONE reason code, ONE action label -- this is one rule, not a bundle of conditional rules.
eligible["reason_code"] = "high_demand_zero_ai_sessions"
eligible["action"] = "review_for_ai_visibility"

eligible["rank"] = eligible["action_score"].rank(method="first", ascending=False).astype(int)
queue = eligible.sort_values("rank")[[
    "rank", "content_hash_id", "client_hash_id", "action_score", "reason_code", "action",
    "total_gsc_impressions_month", "avg_gsc_position_month", "days_with_impressions_month",
    "days_since_update_month", "word_count", "content_type",
]]

out_path = "../outputs/baseline_action_score.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
queue.to_csv(out_path, index=False)

print(f"eligible pool: {len(eligible):,} of {len(df):,} content items "
      f"({len(eligible)/len(df):.1%}) -- real demand, zero AI sessions this month")
print(f"wrote {len(queue):,} ranked rows to {out_path}")
queue.head(10)

eligible pool: 29,788 of 90,489 content items (32.9%) -- real demand, zero AI sessions this month
wrote 29,788 ranked rows to ../outputs/baseline_action_score.csv


,rank,content_hash_id,client_hash_id,action_score,reason_code,action,total_gsc_impressions_month,avg_gsc_position_month,days_with_impressions_month,days_since_update_month,word_count,content_type
47170,1,content_eadb33b5df496f4a,client_e547b89c05043229,1.000000,high_demand_zero_ai_sessions,review_for_ai_visibility,617124.0,2.383011,29,0,2753,keyword article
47282,2,content_ec2e0346994fb5a5,client_e547b89c05043229,0.999966,high_demand_zero_ai_sessions,review_for_ai_visibility,245276.0,2.854514,29,0,2581,keyword article
2030,3,content_0e03de7680314cd5,client_e547b89c05043229,0.999933,high_demand_zero_ai_sessions,review_for_ai_visibility,221310.0,2.675217,29,0,2784,keyword article
2028,4,content_4ffe18112a5642e3,client_e547b89c05043229,0.999899,high_demand_zero_ai_sessions,review_for_ai_visibility,186983.0,2.331060,29,0,3097,keyword article
2023,5,content_8d7d99f109e19aa2,client_e547b89c05043229,0.999866,high_demand_zero_ai_sessions,review_for_ai_visibility,181942.0,2.568135,28,0,2895,keyword article
46899,6,content_44f34c0a90047651,client_23a62021009f63c4,0.999832,high_demand_zero_ai_sessions,review_for_ai_visibility,168160.0,7.324954,20,0,3495,keyword article
48213,7,content_df47d1b976106de4,client_23a62021009f63c4,0.999799,high_demand_zero_ai_sessions,review_for_ai_visibility,124727.0,24.123242,29,0,3435,keyword article
7694,8,content_0ec90963d98b97a5,client_20259bd6705d81d4,0.999765,high_demand_zero_ai_sessions,review_for_ai_visibility,119730.0,3.212505,29,0,3731,keyword article
6354,9,content_545bb6cc7081ded3,client_e547b89c05043229,0.999731,high_demand_zero_ai_sessions,review_for_ai_visibility,117501.0,2.630103,26,0,2811,keyword article
47305,10,content_f86f77b3ebdc05ee,client_e547b89c05043229,0.999698,high_demand_zero_ai_sessions,review_for_ai_visibility,105420.0,3.942110,29,0,1114,keyword article


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
top10 = queue.head(20).reset_index(drop=True)
for i, row in top10.iterrows():
    print(f"{row['rank']:>2}. {row['action']} -- {row['content_hash_id']}")
    print(f"    why: {row['total_gsc_impressions_month']:.0f} impressions this month "
          f"(top of the demand pool), avg position {row['avg_gsc_position_month']:.1f}, "
          f"zero AI-referred sessions observed -- reason code: {row['reason_code']}")
    print(f"    what would make it wrong: if this page's zero AI-session reading is a GA4 "
          f"tracking gap for this client rather than a real absence of AI-referral traffic, "
          f"or if {row['content_type']} pages structurally under-report AI sessions for a "
          f"reason unrelated to content quality")
    print()
top10

 1. review_for_ai_visibility -- content_eadb33b5df496f4a
    why: 617124 impressions this month (top of the demand pool), avg position 2.4, zero AI-referred sessions observed -- reason code: high_demand_zero_ai_sessions
    what would make it wrong: if this page's zero AI-session reading is a GA4 tracking gap for this client rather than a real absence of AI-referral traffic, or if keyword article pages structurally under-report AI sessions for a reason unrelated to content quality

 2. review_for_ai_visibility -- content_ec2e0346994fb5a5
    why: 245276 impressions this month (top of the demand pool), avg position 2.9, zero AI-referred sessions observed -- reason code: high_demand_zero_ai_sessions
    what would make it wrong: if this page's zero AI-session reading is a GA4 tracking gap for this client rather than a real absence of AI-referral traffic, or if keyword article pages structurally under-report AI sessions for a reason unrelated to content quality

 3. review_for_ai_visibili

,rank,content_hash_id,client_hash_id,action_score,reason_code,action,total_gsc_impressions_month,avg_gsc_position_month,days_with_impressions_month,days_since_update_month,word_count,content_type
0,1,content_eadb33b5df496f4a,client_e547b89c05043229,1.000000,high_demand_zero_ai_sessions,review_for_ai_visibility,617124.0,2.383011,29,0,2753,keyword article
1,2,content_ec2e0346994fb5a5,client_e547b89c05043229,0.999966,high_demand_zero_ai_sessions,review_for_ai_visibility,245276.0,2.854514,29,0,2581,keyword article
2,3,content_0e03de7680314cd5,client_e547b89c05043229,0.999933,high_demand_zero_ai_sessions,review_for_ai_visibility,221310.0,2.675217,29,0,2784,keyword article
3,4,content_4ffe18112a5642e3,client_e547b89c05043229,0.999899,high_demand_zero_ai_sessions,review_for_ai_visibility,186983.0,2.331060,29,0,3097,keyword article
4,5,content_8d7d99f109e19aa2,client_e547b89c05043229,0.999866,high_demand_zero_ai_sessions,review_for_ai_visibility,181942.0,2.568135,28,0,2895,keyword article
5,6,content_44f34c0a90047651,client_23a62021009f63c4,0.999832,high_demand_zero_ai_sessions,review_for_ai_visibility,168160.0,7.324954,20,0,3495,keyword article
6,7,content_df47d1b976106de4,client_23a62021009f63c4,0.999799,high_demand_zero_ai_sessions,review_for_ai_visibility,124727.0,24.123242,29,0,3435,keyword article
7,8,content_0ec90963d98b97a5,client_20259bd6705d81d4,0.999765,high_demand_zero_ai_sessions,review_for_ai_visibility,119730.0,3.212505,29,0,3731,keyword article
8,9,content_545bb6cc7081ded3,client_e547b89c05043229,0.999731,high_demand_zero_ai_sessions,review_for_ai_visibility,117501.0,2.630103,26,0,2811,keyword article
9,10,content_f86f77b3ebdc05ee,client_e547b89c05043229,0.999698,high_demand_zero_ai_sessions,review_for_ai_visibility,105420.0,3.942110,29,0,1114,keyword article


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# Weak-pick scan: pages that score high on volume alone but look weak on a signal the score
# ignores -- e.g. already ranking on page 1 (avg_gsc_position_month <= 10), where "AI can't find
# it" is a less likely explanation than "AI just hasn't referred a click yet."
weak = top10[top10["avg_gsc_position_month"] <= 10]
print(f"{len(weak)} of the top 10 already rank in the top 10 on Google search "
      f"(avg_gsc_position_month <= 10) despite zero AI sessions -- worth a second look: is the "
      f"absence of AI referral about the CONTENT, or just about which AI tools happened to get "
      f"asked a matching question this month?")
weak[["rank", "content_hash_id", "avg_gsc_position_month", "total_gsc_impressions_month"]]

# Leakage check: confirm none of the inputs to action_score are future-window or label-derived.
print("\nleakage check:")
print("- total_gsc_impressions_month: summed from the SAME month=2026-03 partition, GSC system, "
      "independent of AI-referral tracking -- not future, not label-derived. OK.")
print("- has_ai_sessions_month: used only to build the ELIGIBLE POOL (filter), never inside the "
      "score itself -- the score is impressions rank alone. OK, same evidence-variable-only rule "
      "as ML-04.")
print("- No FlyRank product flag, priority_score, or health_score was used anywhere above.")

16 of the top 10 already rank in the top 10 on Google search (avg_gsc_position_month <= 10) despite zero AI sessions -- worth a second look: is the absence of AI referral about the CONTENT, or just about which AI tools happened to get asked a matching question this month?

leakage check:
- total_gsc_impressions_month: summed from the SAME month=2026-03 partition, GSC system, independent of AI-referral tracking -- not future, not label-derived. OK.
- has_ai_sessions_month: used only to build the ELIGIBLE POOL (filter), never inside the score itself -- the score is impressions rank alone. OK, same evidence-variable-only rule as ML-04.
- No FlyRank product flag, priority_score, or health_score was used anywhere above.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.